<a href="https://colab.research.google.com/github/vishnubyneni/MLA0202-MACHINE-LEARNING/blob/main/Vindiata_case_study(python_code).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')


* LOAD DATA

In [ ]:
from openpyxl import load_workbook

wb = load_workbook('Analytics Position Case Study (1).xlsx', read_only=True)

def load_sheet(ws, header_col0):
    rows, found = [], False
    for row in ws.iter_rows(values_only=True):
        if row[0] == header_col0:
            found = True; continue
        if found and row[0] is not None:
            rows.append(row)
    return rows

df_games = pd.DataFrame(load_sheet(wb['User Gameplay data'], 'User ID'),
                         columns=['User_ID', 'Games_Played', 'Datetime'])
df_dep   = pd.DataFrame(load_sheet(wb['Deposit Data'],       'User Id'),
                         columns=['User_ID', 'Datetime', 'Amount'])
df_wd    = pd.DataFrame(load_sheet(wb['Withdrawal Data'],    'User Id'),
                         columns=['User_ID', 'Datetime', 'Amount'])

for df in [df_games, df_dep, df_wd]:
    df['User_ID']  = df['User_ID'].astype(int)
    df['Datetime'] = pd.to_datetime(df['Datetime'])

print("Gameplay rows :", len(df_games))
print("Deposit rows  :", len(df_dep))
print("Withdrawal rows:", len(df_wd))
print("Date range    :", df_games['Datetime'].min(), "→", df_games['Datetime'].max())

Gameplay rows : 355266
Deposit rows  : 17438
Withdrawal rows: 3566
Date range    : 2022-01-10 00:00:00 → 2022-12-10 23:59:00


# PART-A

* Functioin Creation

In [ ]:
def filter_slot(df, month, day, slot):
    mask = (df['Datetime'].dt.month == month) & (df['Datetime'].dt.day == day)
    df_d = df[mask].copy()
    return df_d[df_d['Datetime'].dt.hour < 12] if slot == 'S1' else df_d[df_d['Datetime'].dt.hour >= 12]

def calc_loyalty(games_df, dep_df, wd_df, month, day, slot):
    g = filter_slot(games_df, month, day, slot)
    d = filter_slot(dep_df,   month, day, slot)
    w = filter_slot(wd_df,    month, day, slot)

    games_sum = g.groupby('User_ID')['Games_Played'].sum().rename('Games')
    dep_agg   = d.groupby('User_ID').agg(Dep_Amount=('Amount','sum'),
                                          Dep_Count=('Amount','count'))
    wd_agg    = w.groupby('User_ID').agg(Wd_Amount=('Amount','sum'),
                                          Wd_Count=('Amount','count'))

    all_ids = set(games_sum.index) | set(dep_agg.index) | set(wd_agg.index)
    df_out  = pd.DataFrame({'User_ID': list(all_ids)}).set_index('User_ID')
    df_out  = df_out.join(games_sum).join(dep_agg).join(wd_agg).fillna(0)

    df_out['Dep_Diff'] = (df_out['Dep_Count'] - df_out['Wd_Count']).clip(lower=0)
    df_out['Loyalty_Points'] = (
          0.01  * df_out['Dep_Amount']
        + 0.005 * df_out['Wd_Amount']
        + 0.001 * df_out['Dep_Diff']
        + 0.2   * df_out['Games']
    ).round(4)
    return df_out.reset_index().sort_values('Loyalty_Points', ascending=False)

print("Helper functions defined ✓")

Helper functions defined ✓


In [ ]:
# Q1(a) – 2nd October, Slot S1


slot_oct2_s1 = calc_loyalty(df_games, df_dep, df_wd, month=10, day=2, slot='S1')
if slot_oct2_s1.empty:
    print("No activity recorded in October 2 S1.")
else:
    print(f"Players active: {len(slot_oct2_s1)}")
    print(slot_oct2_s1.head(10).to_string(index=False))


# Q1(b) – 16th October, Slot S2


slot_oct16_s2 = calc_loyalty(df_games, df_dep, df_wd, month=10, day=16, slot='S2')
print(f"Players active: {len(slot_oct16_s2)}")
print(slot_oct16_s2[['User_ID','Games','Dep_Amount','Dep_Count','Wd_Amount','Wd_Count','Dep_Diff','Loyalty_Points']].head(15).to_string(index=False))


# Q1(c) – 18th October, Slot S1


slot_oct18_s1 = calc_loyalty(df_games, df_dep, df_wd, month=10, day=18, slot='S1')
print(f"Players active: {len(slot_oct18_s1)}")
print(slot_oct18_s1[['User_ID','Games','Dep_Amount','Dep_Count','Wd_Amount','Wd_Count','Dep_Diff','Loyalty_Points']].head(15).to_string(index=False))


# Q1(d) – 26th October, Slot S2


slot_oct26_s2 = calc_loyalty(df_games, df_dep, df_wd, month=10, day=26, slot='S2')
print(f"Players active: {len(slot_oct26_s2)}")
print(slot_oct26_s2[['User_ID','Games','Dep_Amount','Dep_Count','Wd_Amount','Wd_Count','Dep_Diff','Loyalty_Points']].head(15).to_string(index=False))



In [ ]:
 #Q2 – Overall October Rankings
# Filter October data

oct_g  = df_games[df_games['Datetime'].dt.month == 10]
oct_d  = df_dep  [df_dep  ['Datetime'].dt.month == 10]
oct_w  = df_wd   [df_wd   ['Datetime'].dt.month == 10]

games_sum = oct_g.groupby('User_ID')['Games_Played'].sum().rename('Games')
dep_agg   = oct_d.groupby('User_ID').agg(Dep_Amount=('Amount','sum'), Dep_Count=('Amount','count'))
wd_agg    = oct_w.groupby('User_ID').agg(Wd_Amount=('Amount','sum'),  Wd_Count=('Amount','count'))

all_ids  = set(games_sum.index) | set(dep_agg.index) | set(wd_agg.index)
df_oct   = pd.DataFrame({'User_ID': list(all_ids)}).set_index('User_ID')
df_oct   = df_oct.join(games_sum).join(dep_agg).join(wd_agg).fillna(0).reset_index()

df_oct['Dep_Diff'] = (df_oct['Dep_Count'] - df_oct['Wd_Count']).clip(lower=0)
df_oct['Loyalty_Points'] = (
      0.01  * df_oct['Dep_Amount']
    + 0.005 * df_oct['Wd_Amount']
    + 0.001 * df_oct['Dep_Diff']
    + 0.2   * df_oct['Games']
).round(4)

df_oct = df_oct.sort_values(['Loyalty_Points', 'Games'], ascending=[False, False]).reset_index(drop=True)
df_oct['Rank'] = df_oct.index + 1

print(f"Total players in October: {len(df_oct)}")
print("\nTop 20 Players:")
print(df_oct[['Rank','User_ID','Games','Dep_Amount','Wd_Amount','Loyalty_Points']].head(20).to_string(index=False))


In [ ]:
# Q3 – Average Deposit Amount (Overall)
avg_deposit = df_dep['Amount'].mean()
print(f"Average deposit amount (all transactions): Rs {avg_deposit:,.2f}")

In [ ]:

# Q4 – Average Deposit Amount Per User Per Month

user_month_dep = df_dep.copy()
user_month_dep['Month'] = user_month_dep['Datetime'].dt.to_period('M')
monthly_totals = user_month_dep.groupby(['User_ID','Month'])['Amount'].sum()
avg_per_user_month = monthly_totals.mean()
print(f"Average deposit per user per month: Rs {avg_per_user_month:,.2f}"

In [ ]:
# Q5 – Average Number of Games Played Per User

# Total games per user
total_games_per_user = df_games.groupby('User_ID')['Games_Played'].sum()
avg_games = total_games_per_user.mean()
print(f"Average total games per user (full year): {avg_games:,.2f}")

# Monthly average per user
df_games['Month'] = df_games['Datetime'].dt.to_period('M')
monthly_games = df_games.groupby(['User_ID','Month'])['Games_Played'].sum()
avg_monthly_games = monthly_games.groupby('User_ID').mean().mean()
print(f"Average games per user per month: {avg_monthly_games:,.2f}")

# PART-B

In [ ]:
top50 = df_oct.head(50).copy()
total_bonus = 50_000

# Tiered bonus
def tiered_bonus(rank):
    if rank <= 10: return 2000
    elif rank <= 25: return 1000
    else: return 600

top50['Tiered_Bonus'] = top50['Rank'].apply(tiered_bonus)

# LP-proportional bonus (for comparison)
top50['LP_Proportional_Bonus'] = (top50['Loyalty_Points'] / top50['Loyalty_Points'].sum() * total_bonus).round(2)

print("Bonus Pool Verification:", top50['Tiered_Bonus'].sum(), "(should be 50000)")
print()
print(top50[['Rank','User_ID','Games','Dep_Amount','Wd_Amount','Loyalty_Points',
             'Tiered_Bonus','LP_Proportional_Bonus']].to_string(index=False))


# Visualise both approaches
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title in zip(
    axes,
    ['Tiered_Bonus', 'LP_Proportional_Bonus'],
    ['Tiered Bonus', 'LP-Proportional Bonus']
):
    ax.bar(top50['Rank'], top50[col], color='steelblue', edgecolor='white', linewidth=0.5)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel("Player Rank")
    ax.set_ylabel("Bonus (Rs)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.suptitle("Bonus Distribution Comparison – Top 50 Players", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('bonus_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved.")


# PART-C

In [ ]:
# Show LP component breakdown for top 20
top20 = df_oct.head(20).copy()
top20['LP_from_Deposit'] = 0.01  * top20['Dep_Amount']
top20['LP_from_Withdraw'] = 0.005 * top20['Wd_Amount']
top20['LP_from_DiffBonus'] = 0.001 * top20['Dep_Diff']
top20['LP_from_Games'] = 0.2 * top20['Games']

print("LP Component Breakdown (Top 20):")
print(top20[['Rank','User_ID','LP_from_Deposit','LP_from_Withdraw','LP_from_Games','LP_from_DiffBonus','Loyalty_Points']].to_string(index=False))

# Show how much of LP comes from money (deposit+withdrawal) vs games
top20['Pct_Money'] = (top20['LP_from_Deposit'] + top20['LP_from_Withdraw']) / top20['Loyalty_Points'] * 100
top20['Pct_Games'] = top20['LP_from_Games'] / top20['Loyalty_Points'] * 100
print(f"\nAvg % of LP from monetary activity (top 20): {top20['Pct_Money'].mean():.1f}%")
print(f"Avg % of LP from games played   (top 20): {top20['Pct_Games'].mean():.1f}%")


# Visualise LP component split for top 10
top10 = df_oct.head(10).copy()
top10['LP_from_Deposit']   = 0.01  * top10['Dep_Amount']
top10['LP_from_Withdraw']  = 0.005 * top10['Wd_Amount']
top10['LP_from_DiffBonus'] = 0.001 * top10['Dep_Diff']
top10['LP_from_Games']     = 0.2   * top10['Games']

fig, ax = plt.subplots(figsize=(12, 5))
labels = [f"U{uid}\n(R{rank})" for uid, rank in zip(top10['User_ID'], top10['Rank'])]
bottom = np.zeros(10)
for col, label, color in [
    ('LP_from_Deposit',   'Deposit LP',   '#4e79a7'),
    ('LP_from_Withdraw',  'Withdrawal LP','#f28e2b'),
    ('LP_from_Games',     'Games LP',     '#59a14f'),
    ('LP_from_DiffBonus', 'Count Diff LP','#e15759'),
]:
    ax.bar(labels, top10[col], bottom=bottom, label=label, color=color)
    bottom += top10[col].values

ax.set_title("LP Component Breakdown – Top 10 Players (October)", fontweight='bold')
ax.set_ylabel("Loyalty Points")
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('lp_components.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved.")